In [17]:
from langgraph.runtime import Runtime
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.runnables.config import RunnableConfig
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

In [18]:
class AgentState(MessagesState):
    intent: str
    order_id: str
    preferred_language: str

In [19]:
class Context(TypedDict):
    user_id: str
    plan: str

In [20]:
def parse_request(state: AgentState, config: RunnableConfig) -> dict:
    user_text = str(state["messages"][-1].content).strip()
    thread_id = config["configurable"]["thread_id"]

    print(f"[Node 1] thread_id = {thread_id}")
    print(f"[Node 1] message_count = {len(state['messages'])}")
    print(f"[Node 1] existing_order = {state.get('order_id', 'N/A')}") 

    if user_text.startswith("My order ID is"):
        order_id = user_text.removeprefix("My order ID is ").strip()

        return {
            "order_id": order_id,
            "intent": "save_order"
        }

    if user_text == "Where is my order?":
        return {"intent": "ask_order"}

    if user_text.startswith("Remember my language is "):
        return {"intent": "save_language"}

    if user_text == "Which language do I prefer?":
        return {"intent": "ask_language"}

    return {"intent": "unknown"}

In [21]:
def load_user_memory(state: AgentState, runtime: Runtime[Context]) -> dict:
    user_id = runtime.context.get("user_id")
    plan = runtime.context.get("plan")

    store = runtime.store

    namespace = ("users", user_id)

    if state["intent"] == "save_language":
        message = str(state["messages"][-1].content).strip()
        language = message.removeprefix("Remember my language is ").strip()
        store.put(
            namespace,
            "language",
            {"value": language}
        )

    memory = store.get(namespace, "language")

    language = (
        memory.value["value"]
        if memory is not None else "not set"
    )

    return {
        "preferred_language": language
    }

In [22]:
def respond(
        state: AgentState,
        config: RunnableConfig,
        runtime: Runtime[Context]
) -> dict:
    thread_id = config['configurable']["thread_id"]
    user_id = runtime.context.get("user_id")

    intent = state["intent"]

    if intent == "save_order":
        answer = (
            f"Saved order {state['order_id']} for user {user_id} in thread {thread_id}"
        )

    elif intent == "ask_order":
        if state.get("order_id"):
            answer = (
                f"Your order {state['order_id']} is being processed. "
                "It will be shipped soon."
            )
        else:
            answer = (
                "I couldn't find your order. Please provide your order ID."
            )

    elif intent == "save_language":
        answer = (
            f"Saved your preferred language {state['preferred_language']}"
        )
        
    elif intent == "ask_language":
        if state.get("preferred_language"):
            answer = (
                f"Your preferred language is {state['preferred_language']}"
            )
        else:
            answer = (
                "You haven't set your preferred language yet. "
                "Please set your preferred language."
            )

    else:
        answer = "Ask about your order or language preferences."

    return {
        "messages": [AIMessage(content=answer)]
    }

In [23]:
builder = StateGraph(AgentState, context_schema=Context)

builder.add_node("parse_request", parse_request)
builder.add_node("load_user_memory", load_user_memory)
builder.add_node("respond", respond)

builder.add_edge(START, "parse_request")
builder.add_edge("parse_request", "load_user_memory")
builder.add_edge("load_user_memory", "respond")
builder.add_edge("respond", END)

checkpointer = InMemorySaver()
store = InMemoryStore()

graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

In [24]:
def chat(user_id, thread_id, message, plan="free"):
    print(f"\nUser {user_id} | Thread {thread_id}")
    print(f"\nInput: {message}")

    result = graph.invoke({
        "messages": [HumanMessage(content=message)]
    },
    config={
        "configurable": {
            "thread_id": thread_id
        },
        "tags": ["chat", "demo"]
    },
    context={
        "user_id": user_id,
        "plan": plan
    })

    # graph has been invoked successfully
    print("Answer: ", result["messages"][-1].content)

    print("Total messages: ", len(result["messages"]))

    return result

In [25]:
chat("user_a", "a1", "My order ID is 1234567890", "premium")


User user_a | Thread a1

Input: My order ID is 1234567890
[Node 1] thread_id = a1
[Node 1] message_count = 1
[Node 1] existing_order = N/A
Answer:  Saved order 1234567890 for user user_a in thread a1
Total messages:  2


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='1f70e64d-5e61-4f95-9b5e-f4f26fd20c9e'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='9bbea862-15e1-40e7-b941-af51410fa912', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'save_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [26]:
chat("user_a", "a1", "Where is my order?", "premium")


User user_a | Thread a1

Input: Where is my order?
[Node 1] thread_id = a1
[Node 1] message_count = 3
[Node 1] existing_order = 1234567890
Answer:  Your order 1234567890 is being processed. It will be shipped soon.
Total messages:  4


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='1f70e64d-5e61-4f95-9b5e-f4f26fd20c9e'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='9bbea862-15e1-40e7-b941-af51410fa912', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='4a99f20f-5930-4584-9361-7e096fa96502'),
  AIMessage(content='Your order 1234567890 is being processed. It will be shipped soon.', additional_kwargs={}, response_metadata={}, id='19ef8eb0-5f72-425e-8a4f-3e64526513bd', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'order_id': '1234567890',
 'preferred_language': 'not set'}

In [27]:
chat("user_a", "a1", "Remember my language is German", "premium")


User user_a | Thread a1

Input: Remember my language is German
[Node 1] thread_id = a1
[Node 1] message_count = 5
[Node 1] existing_order = 1234567890
Answer:  Saved your preferred language German
Total messages:  6


{'messages': [HumanMessage(content='My order ID is 1234567890', additional_kwargs={}, response_metadata={}, id='1f70e64d-5e61-4f95-9b5e-f4f26fd20c9e'),
  AIMessage(content='Saved order 1234567890 for user user_a in thread a1', additional_kwargs={}, response_metadata={}, id='9bbea862-15e1-40e7-b941-af51410fa912', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='4a99f20f-5930-4584-9361-7e096fa96502'),
  AIMessage(content='Your order 1234567890 is being processed. It will be shipped soon.', additional_kwargs={}, response_metadata={}, id='19ef8eb0-5f72-425e-8a4f-3e64526513bd', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Remember my language is German', additional_kwargs={}, response_metadata={}, id='a836dd0c-0b2a-49dd-b701-486f1478504e'),
  AIMessage(content='Saved your preferred language German', additional_kwargs={}, response_metadata={}, id='4f06b035-4210-4639-8522-44c4d3e1d6af', tool_c

In [28]:
chat("user_a", "a2", "Where is my order?", "premium")


User user_a | Thread a2

Input: Where is my order?
[Node 1] thread_id = a2
[Node 1] message_count = 1
[Node 1] existing_order = N/A
Answer:  I couldn't find your order. Please provide your order ID.
Total messages:  2


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='e4084554-361c-4b99-afd9-e0ff75c45f14'),
  AIMessage(content="I couldn't find your order. Please provide your order ID.", additional_kwargs={}, response_metadata={}, id='69380e92-0ecb-4a92-8b51-745605f9fbf8', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_order',
 'preferred_language': 'German'}

In [29]:
chat("user_a", "a2", "Which language do I prefer?", "premium")


User user_a | Thread a2

Input: Which language do I prefer?
[Node 1] thread_id = a2
[Node 1] message_count = 3
[Node 1] existing_order = N/A
Answer:  Your preferred language is German
Total messages:  4


{'messages': [HumanMessage(content='Where is my order?', additional_kwargs={}, response_metadata={}, id='e4084554-361c-4b99-afd9-e0ff75c45f14'),
  AIMessage(content="I couldn't find your order. Please provide your order ID.", additional_kwargs={}, response_metadata={}, id='69380e92-0ecb-4a92-8b51-745605f9fbf8', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Which language do I prefer?', additional_kwargs={}, response_metadata={}, id='36896cd4-15b4-4c28-8d5a-3a5fe055a3c3'),
  AIMessage(content='Your preferred language is German', additional_kwargs={}, response_metadata={}, id='5a5c97a3-1a2a-4357-9f67-16787449ed5a', tool_calls=[], invalid_tool_calls=[])],
 'intent': 'ask_language',
 'preferred_language': 'German'}